1. 데이터 로드
2. 데이터 더미 변수화
    - -9 -> nan -> 더미 변수도 nan
3. 각 변수별 상관관계 측정 (피어슨 상관계수)

In [14]:
import os
import pandas as pd
import numpy as np

'''CURDIR = os.path.dirname(__file__)
print(CURDIR)'''
CURDIR = os.getcwd()
DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
DATA = pd.read_csv(DATA_PATH)

# 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
# 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
for col in DATA.columns:
    cats = list(DATA[col].unique())
    DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

In [15]:
# validation set, test set이 imputation 설계하는 데 들어가면 안 됨, 일종의 사후판단 정보가 들어갈 수 있기 때문
from sklearn.model_selection import train_test_split
y = DATA['REASONb']
X = DATA.drop('REASONb', axis=1)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# X_train만 가지고 imputation 모델 설계, 이후 X_val, X_test에 해당 모델 적용

In [16]:
temp_df = X_train.copy()
temp_df = temp_df.where(temp_df != -9, np.nan)
for col in temp_df.columns:
    try:
        temp_df[col] = temp_df[col].cat.remove_categories(-9)
    except ValueError:
        continue

In [17]:
mask = temp_df.isna()
data_dum = pd.get_dummies(temp_df, dtype=int)
dum_temp = data_dum.copy()

for target_col in mask.columns:
    mask_specified = mask[target_col]
    target_col = target_col + '_'
    object_cols = [i for i in data_dum.columns if target_col in i]
    dum_temp.loc[mask_specified, object_cols] = np.nan
    
    
mask_dum = ~dum_temp.isna() # 결측일 때 False가 되게
mask_dum = mask_dum.astype(int)
mask_dum

,STFIPS_2,STFIPS_51,STFIPS_8,STFIPS_46,STFIPS_6,STFIPS_45,STFIPS_9,STFIPS_26,STFIPS_23,STFIPS_44,...,CBSA2020_49260,CBSA2020_49300,CBSA2020_49340,CBSA2020_49380,CBSA2020_49420,CBSA2020_49460,CBSA2020_49620,CBSA2020_49660,CBSA2020_49700,CBSA2020_49740
1139344,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
847019,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1018611,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
1340597,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1392195,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110268,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
259178,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
131932,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
671155,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [5]:
'''pearson_dum = data_dum.corr()
pearson_dum.to_csv("pearson_dum.csv")
pearson_dum'''
pearson_dum = pd.read_csv("pearson_dum.csv", index_col=0)
pearson_dum = pearson_dum.fillna(0)
# 5시간 걸려서 만든 데이터프레임에 결측치가 있었음 -> 일단 0으로 채우고 진행
pearson_dum

,STFIPS_2,STFIPS_51,STFIPS_8,STFIPS_46,STFIPS_6,STFIPS_45,STFIPS_9,STFIPS_26,STFIPS_23,STFIPS_44,...,CBSA2020_13220,CBSA2020_48260,CBSA2020_37620,CBSA2020_16220,CBSA2020_23940,CBSA2020_40540,CBSA2020_40180,CBSA2020_27220,CBSA2020_16940,CBSA2020_43260
STFIPS_2,1.000000,-0.006261,-0.012313,-0.006013,-0.017920,-0.009313,-0.010565,-0.013150,-0.002850,-0.005052,...,-0.000764,-0.000124,-0.000380,-0.000547,-0.000643,-0.000877,-0.000880,-0.000682,-0.000380,-0.000819
STFIPS_51,-0.006261,1.000000,-0.020711,-0.010114,-0.030143,-0.015666,-0.017772,-0.022120,-0.004794,-0.008499,...,-0.000553,-0.000253,-0.000774,-0.001114,-0.001308,-0.001785,-0.001791,-0.001387,-0.000774,-0.001667
STFIPS_8,-0.012313,-0.020711,1.000000,-0.019889,-0.059276,-0.030808,-0.034949,-0.043500,-0.009428,-0.016713,...,-0.002679,-0.000436,-0.001333,-0.001919,-0.002254,-0.002036,-0.002050,-0.001722,-0.001333,-0.002317
STFIPS_46,-0.006013,-0.010114,-0.019889,1.000000,-0.028946,-0.015044,-0.017067,-0.021242,-0.004604,-0.008161,...,-0.001530,-0.000249,-0.000762,-0.001096,-0.001287,-0.001757,-0.000880,-0.001366,-0.000762,-0.001641
STFIPS_6,-0.017920,-0.030143,-0.059276,-0.028946,1.000000,-0.044838,-0.050865,-0.063310,-0.013721,-0.024324,...,-0.004570,-0.000745,-0.002275,-0.002223,-0.003845,-0.005248,-0.004939,-0.004078,-0.002275,-0.004200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CBSA2020_40540,-0.000877,-0.001785,-0.002036,-0.001757,-0.005248,-0.002728,-0.003186,-0.003736,-0.000676,-0.000539,...,-0.000199,-0.000032,-0.000099,-0.000143,-0.000168,1.000000,-0.000230,-0.000178,-0.000099,-0.000214
CBSA2020_40180,-0.000880,-0.001791,-0.002050,-0.000880,-0.004939,-0.002737,-0.003197,-0.003748,-0.000678,-0.000541,...,-0.000200,-0.000033,-0.000100,-0.000143,-0.000168,-0.000230,1.000000,-0.000178,-0.000100,-0.000214
CBSA2020_27220,-0.000682,-0.001387,-0.001722,-0.001366,-0.004078,-0.002120,-0.002476,-0.002903,-0.000525,-0.000419,...,-0.000155,-0.000025,-0.000077,-0.000111,-0.000130,-0.000178,-0.000178,1.000000,-0.000077,-0.000166
CBSA2020_16940,-0.000380,-0.000774,-0.001333,-0.000762,-0.002275,-0.001182,-0.001381,-0.001619,-0.000293,-0.000234,...,-0.000086,-0.000014,-0.000043,-0.000062,-0.000073,-0.000099,-0.000100,-0.000077,1.000000,-0.000093


In [6]:
import pandas as pd
import numpy as np

def get_prefix(col):
    if '_D' in col:
        col_split_ = col.split('_')
        prefix = col_split_[0] + '_D'
    else:
        col_split_ = col.split('_')
        prefix = col_split_[0]
    return prefix

def get_prefix_dict(data_dum):
    cols = data_dum.columns
    if not cols.any():
        return {}
        
    prefix_dict = {}
    dict_elem = []
    cur_prefix = get_prefix(cols[0])
    
    for count, col in enumerate(cols):
        new_prefix = get_prefix(col)
        
        if new_prefix == cur_prefix:
            dict_elem.append(count)
        else:
            prefix_dict[cur_prefix] = dict_elem
            cur_prefix = new_prefix
            dict_elem = [count] 
    if dict_elem:
        prefix_dict[cur_prefix] = dict_elem
        
    return prefix_dict


In [7]:
def get_distance(w, a_v, a_i, a_j, mask_dum, pearson_dum, data_dum):
    '''
    a_v: 원래 데이터에서 결측치가 속한 변수명
    a_i: 원래 데이터 (같은 값이지만 더미 데이터)의 결측치가 속한 행
    결측치 위치 중 행 번호: a_i, 열 번호: a_v

    a_j: 원래 데이터 (같은 값이지만 더미 데이터)의 거리를 재고자 하는 행
    '''
    mask_i = mask_dum.iloc[a_i]
    mask_j = mask_dum.iloc[a_j]
    mask_common = mask_i * mask_j

    a_ij = sum(mask_common)

    vec_i = data_dum.iloc[a_i]
    vec_j = data_dum.iloc[a_j]
    vec_diff = (vec_i - vec_j) ** 2

    prefix_dict = get_prefix_dict(data_dum)
    r_vec = np.mean(pearson_dum.iloc[prefix_dict[a_v]], axis=0)
    r_vec = abs(r_vec) ** w

    return pow((1 / a_ij) * sum(vec_diff * mask_common * r_vec), 0.5)


In [8]:
def gaussian(distance, tau):
    if tau == 0:
        u = np.divide(distance, tau, out=np.zeros_like(distance), where=tau!=0)
    else:
        u = distance / tau
    constant = 1.0 / np.sqrt(2.0 * np.pi)
    K_u = constant * np.exp(-0.5 * u**2)
    return K_u

In [9]:
def weighting_scheme(w, tau, a_v, target_idx, mask_dum, pearson_dum, data_dum):
    # 타겟 행과 타겟 행을 제외한 모든 행에 대해 거리 계산
    # a_v: 원래 데이터에서 결측치가 속한 변수명
    dist_list = []
    index_list = list(data_dum.index)
    if target_idx in index_list:
        index_list.remove(target_idx)
    for i in index_list:
        dist_list.append(get_distance(w, a_v, target_idx, i, mask_dum, pearson_dum,data_dum))

    # 가중치 계산
    dist_np = np.array(dist_list)
    weights = gaussian(dist_np, tau)
    weights = weights / sum(weights)
    return weights

In [10]:
def impute_cell(w, tau, a_v, target_idx, mask_dum, pearson_dum, data_dum):
    '''
    imputation estimator (pi)
    argmax -> impute value
    '''
    a_v = a_v + '_'
    col_list = [i for i in data_dum.columns if a_v in data_dum.columns]
    sliced_df = data_dum.loc[:, col_list]
    sliced_df.drop(index=target_idx)
    sliced_m = sliced_df.to_numpy()

    weights = weighting_scheme(w, tau, a_v, target_idx, mask_dum, pearson_dum, data_dum)
    
    weighted_sliced_m = sliced_m * weights[:, np.newaxis]
    vec = np.sum(weighted_sliced_m, axis=0)
    vec = vec / sum(vec)

    result = col_list[np.argmax(vec)]
    result = result.lstrip(a_v)
    return result

In [11]:
def impute(w, tau, mask_dum, pearson_dum, data_dum, mask, raw_df):
    df = raw_df.copy()

    # 최적화: 결측치가 있는 행만 순회합니다.
    missing_rows = mask[mask.eq(0).any(axis=1)]

    for i in missing_rows.index: # i = 결측이 있는 행 인덱스
        target_vec = mask.loc[i]
        target_cols = target_vec[target_vec==0].index # 결측인 변수명 리스트

        for col in target_cols: # col = 결측인 변수명 (예: 'STFIPS')
            value_impute = impute_cell(w, tau, col, i, mask_dum, pearson_dum, data_dum)

            # [수정됨] target_col (전체)이 아닌 col (특정 변수)에 값을 대입합니다.
            df.loc[i, col] = value_impute 
    return df

In [14]:
# (get_prefix, get_prefix_dict, gaussian, get_distance 함수는 셀 88, 89, 90에 정의되어 있다고 가정)
# [수정] get_distance는 prefix_dict를 인자로 받도록 변경
def get_distance(w, a_v_prefix, a_i, a_j, mask_dum, pearson_dum, data_dum, prefix_dict):
    '''
    a_v_prefix: 원래 데이터에서 결측치가 속한 변수명 (예: 'STFIPS')
    a_i: 결측치가 속한 행 인덱스
    a_j: 거리를 재고자 하는 행 인덱스
    prefix_dict: 미리 계산된 접두사 맵
    '''
    mask_i = mask_dum.loc[a_i]
    mask_j = mask_dum.loc[a_j]
    mask_common = mask_i * mask_j

    a_ij = np.sum(mask_common)
    
    if a_ij == 0:
        return np.inf # 공동 관측치가 없으면 무한대 거리

    vec_i = data_dum.loc[a_i]
    vec_j = data_dum.loc[a_j]
    vec_diff = (vec_i - vec_j) ** 2

    # [수정] a_v_prefix (예: 'STFIPS')를 키로 사용
    r_vec = np.mean(pearson_dum.iloc[prefix_dict[a_v_prefix]], axis=0)
    r_vec = np.abs(r_vec) ** w

    # 0으로 나누기 방지
    if a_ij == 0:
        return np.inf
        
    return np.sqrt((1 / a_ij) * np.sum(vec_diff * mask_common * r_vec))

# [수정] weighting_scheme은 prefix_dict를 인자로 받도록 변경
def weighting_scheme(w, tau, a_v_prefix, target_idx, mask_dum, pearson_dum, data_dum, prefix_dict):
    dist_list = []
    
    # [수정] target_idx가 아닌 행들만 가져옴
    other_indices = data_dum.index.drop(target_idx)
    
    for i in other_indices:
        dist = get_distance(w, a_v_prefix, target_idx, i, mask_dum, pearson_dum, data_dum, prefix_dict)
        dist_list.append(dist)

    dist_np = np.array(dist_list)
    
    # 0으로 나누기 방지
    if np.sum(dist_np) == 0:
         # 모든 거리가 0이면 균등 가중치
        return np.full_like(dist_np, 1.0 / len(dist_np))

    weights = gaussian(dist_np, tau)
    
    sum_weights = np.sum(weights)
    if sum_weights == 0:
        # 가우시안 결과가 0 (거리가 너무 멈)이면, 거리의 역수에 비례하는 가중치 사용
        inv_dist = 1.0 / (dist_np + 1e-9) # 1e-9는 0으로 나누기 방지
        weights = inv_dist / np.sum(inv_dist)
    else:
        weights = weights / sum_weights
        
    return weights, other_indices

# [수정] impute_cell (오류 수정)
def impute_cell(w, tau, a_v_prefix, target_idx, mask_dum, pearson_dum, data_dum, prefix_dict):
    '''
    a_v_prefix: 결측 변수의 접두사 (예: 'STFIPS')
    '''
    
    # 1. [수정] prefix_dict에서 해당 접두사의 더미 컬럼 인덱스를 가져옴
    target_dummy_indices = prefix_dict.get(a_v_prefix)
    
    if target_dummy_indices is None:
        raise KeyError(f"접두사 '{a_v_prefix}'를 prefix_dict에서 찾을 수 없습니다.")
        
    # 2. [수정] 컬럼 이름 리스트 생성
    col_list = data_dum.columns[target_dummy_indices]
    
    # 3. [수정] target_idx를 제외한 가중치와 인덱스 계산
    weights, other_indices = weighting_scheme(w, tau, a_v_prefix, target_idx, mask_dum, pearson_dum, data_dum, prefix_dict)

    # 4. [수정] other_indices (target_idx가 제외된)를 사용하여 데이터 슬라이싱
    sliced_df = data_dum.loc[other_indices, col_list]
    sliced_m = sliced_df.to_numpy() # (N-1, k) 배열

    # 5. [수정] (N-1, k) * (N-1, 1) -> (N-1, k)
    weighted_sliced_m = sliced_m * weights[:, np.newaxis]
    
    vec = np.sum(weighted_sliced_m, axis=0) # (k,) 벡터
    
    # 6. [수정] 확률 벡터 표준화
    sum_vec = np.sum(vec)
    if sum_vec > 0:
        vec = vec / sum_vec
    else:
        # 모든 가중 합이 0이면 (예: 모든 이웃이 0), 첫 번째 범주 선택
        vec[0] = 1 

    # 7. [수정] 컬럼 이름(col_list)에서 argmax를 찾고 접두사 제거
    imputed_col_name = col_list[np.argmax(vec)]
    
    # 8. [수정] 접두사 제거 로직 (get_prefix 논리 활용)
    prefix_len = len(a_v_prefix)
    if a_v_prefix.endswith('_D'): # 'METRO_D' 같은 경우
        result = imputed_col_name[prefix_len+1:] # 'METRO_D_1' -> '1'
    else: # 'STFIPS' 같은 경우
        result = imputed_col_name[prefix_len+1:] # 'STFIPS_51' -> '51'
        
    return result

# [수정] impute (최종)
def impute(w, tau, mask_dum, pearson_dum, data_dum, mask, raw_df):
    df = raw_df.copy()
    
    # 1. [최적화] prefix_dict를 한 번만 계산
    prefix_dict = get_prefix_dict(data_dum)
    
    # 2. [최적화] 결측치가 있는 행만 순회
    missing_rows = mask[mask.eq(0).any(axis=1)]
    
    print(f"총 {len(missing_rows)}개의 결측 행 대치를 시작합니다...")
    
    count = 0
    for i in missing_rows.index: # i = 결측이 있는 행 인덱스
        target_vec = mask.loc[i]
        target_cols = target_vec[target_vec==0].index # 결측인 변수명 리스트 (예: ['STFIPS', 'METRO_D'])
        
        for col_prefix in target_cols: # col_prefix = 결측인 변수명 (예: 'STFIPS')
            
            # 3. [수정] prefix_dict를 인자로 전달
            value_impute = impute_cell(w, tau, col_prefix, i, mask_dum, pearson_dum, data_dum, prefix_dict)
            
            # 4. [수정] 정확한 셀에 대치
            df.loc[i, col_prefix] = value_impute 
        
        count += 1
        if count % 100 == 0:
            print(f"{count} / {len(missing_rows)} 행 처리 완료...")
            
    print("대치 완료.")
    return df

In [15]:
df = impute(2, 2, mask_dum, pearson_dum, data_dum, mask, temp_df)

총 917983개의 결측 행 대치를 시작합니다...


KeyboardInterrupt: 